# 03 · Analysis

## The question

Subscriber count is the number quoted whenever a YouTube channel is described — in press
coverage, in sponsorship decks, in valuations. **How much does it tell you about how many people
actually watch?**

Linus Media Group is a useful place to ask, because it runs four channels of very different sizes
under one roof. Comparing them to each other removes most of what makes cross-company comparison
unreliable: the same production company, the same era, the same audience pool, no peer group to
select.

## Scope

**The analysis is within the account.** Peers appear once, as background, and no claim rests on
them.

That choice fixes a specific problem. An earlier draft compared LMG against twelve peer channels
and used a 45-minute duration cap to exclude podcasts. The cap turned out to be badly asymmetric —
four peers have no content above it at all, while it removes 15.5% of the flagship's catalogue —
and it moved the headline number by 22% in LMG's favour. Comparing a company against itself makes
the cap unnecessary: **duration becomes a way of classifying content, not a filter applied to it.**

Constraints inherited from `02`: view counts are cumulative, so cross-era comparisons of absolute
levels carry an accrual bias that cannot be identified from cross-sectional data; the rolling
baseline index is a normalisation and cannot carry p-values; intervals use a moving-block bootstrap.

The main result below is a **snapshot** — one window, no comparison across eras — so none of those
constraints bind on it.

In [ ]:
import json, glob, pathlib
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

CWD = pathlib.Path.cwd()
NB = next((p for p in [CWD, CWD/"notebook", *CWD.parents] if (p/"data").is_dir()), CWD)
DATA, FIG = NB/"data", NB/"figures"; FIG.mkdir(exist_ok=True)

d = pd.read_csv(DATA/"videos_clean.csv", parse_dates=["date"])
SUBS = pd.read_csv(DATA/"channels.csv").set_index("handle").subs.to_dict()
ORDER = ["LinusTechTips", "Techquickie", "ShortCircuit", "TechLinked"]
COLOR = dict(zip(ORDER, ["#2E86DE", "#E55039", "#F6A609", "#16A085"]))

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})

END = d.date.max()
recent = d[(d.date >= END - pd.Timedelta(days=730)) & (d.format != "wan_show")].copy()
print(f"{len(d):,} videos on file · analysis window {(END-pd.Timedelta(days=730)).date()} → {END.date()}")
print(f"{len(recent):,} videos in window (podcast excluded)")

## 1. Duration as a classification

Before comparing performance, it is worth seeing what each channel actually makes. Duration bands
are used here to describe the catalogue, not to trim it.

In [ ]:
BANDS = [(0, 1, "Short (<1m)"), (1, 8, "Brief (1–8m)"), (8, 20, "Standard (8–20m)"),
         (20, 45, "Long (20–45m)"), (45, np.inf, "Extended (>45m)")]
BAND_ORDER = [b[2] for b in BANDS]

def band(m):
    for lo, hi, name in BANDS:
        if lo < m <= hi:
            return name
    return BAND_ORDER[0]

recent["band"] = recent.duration_min.map(band)
mix = (recent.pivot_table(index="band", columns="channel", values="id",
                          aggfunc="count", fill_value=0)
             .reindex(BAND_ORDER).reindex(columns=ORDER))
print("Videos published in the last 24 months, by length\n")
print(mix.to_string())
print("\nAs a share of each channel's output\n")
print((mix/mix.sum()).map(lambda v: f"{v:.0%}").to_string())
print("\nEach channel sits in a different band. The network is not one product repeated —")
print("it covers different amounts of a viewer's time.")

## 2. Main result — subscribers barely predict reach, inside one company

One number per channel: the median views of a video published in the last 24 months, divided by
the channel's subscriber count. It asks how many of the people who signed up actually turn up.

Only standard long-form is used, so the comparison is like for like.

In [ ]:
lv = recent[recent.format == "long_video"]

reach = (lv.groupby("channel")
           .agg(videos=("id", "size"),
                median_views=("views", "median"),
                median_minutes=("duration_min", "median"))
           .reindex(ORDER))
reach["subscribers"] = [SUBS[c] for c in reach.index]
reach["views_per_1k_subs"] = (reach.median_views/reach.subscribers*1000).round(0)
reach["like_rate_pct"] = lv.groupby("channel").apply(
    lambda g: g.likes.sum()/g.views.sum()*100, include_groups=False).round(2)
reach = reach.sort_values("views_per_1k_subs", ascending=False)

print(reach.to_string())

hi, lo = reach.views_per_1k_subs.max(), reach.views_per_1k_subs.min()
rank = list(reach.index).index("LinusTechTips") + 1
print(f"\nSpread inside one company: {lo:.0f} to {hi:.0f} — {hi/lo:.1f}x")
print(f"The flagship has {SUBS['LinusTechTips']/SUBS['TechLinked']:.1f}x TechLinked's subscribers "
      f"and ranks {rank} of 4 on reach per subscriber.")
reach.to_csv(DATA/"reach_per_sub_lmg.csv")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.4))

y = np.arange(len(reach))[::-1]
ax[0].barh(y, reach.views_per_1k_subs, color=[COLOR[c] for c in reach.index])
ax[0].set_yticks(y); ax[0].set_yticklabels(reach.index)
ax[0].set_xlabel("Median views per 1,000 subscribers")
ax[0].set_title("The same subscriber is worth very different reach", loc="left", weight="bold")
for yi, v in zip(y, reach.views_per_1k_subs):
    ax[0].text(v+4, yi, f"{v:.0f}", va="center", fontsize=9)

ax[1].scatter(reach.subscribers/1e6, reach.views_per_1k_subs,
              s=140, c=[COLOR[c] for c in reach.index], zorder=3)
for ch, r in reach.iterrows():
    ax[1].annotate(ch, (r.subscribers/1e6, r.views_per_1k_subs),
                   xytext=(8, 4), textcoords="offset points", fontsize=9)
ax[1].set_xscale("log")
ax[1].set_xlabel("Subscribers (millions, log)")
ax[1].set_ylabel("Views per 1,000 subscribers")
ax[1].set_title("Size does not order them", loc="left", weight="bold")
ax[1].set_xlim(1.2, 30)

plt.tight_layout(); plt.savefig(FIG/"subs_vs_reach_lmg.png", bbox_inches="tight"); plt.show()

> **What this says.** Inside a single company, in a single window, the number of people a video
> reaches per subscriber varies more than fourfold, and the largest channel is not the most
> efficient. No control group is needed and no significance test applies — the spread is the
> observation.
>
> **What it does not say.** That a low ratio means a channel is failing. Reach efficiency and
> commercial value are different things: the two channels with the weakest reach per subscriber
> have the *highest* like rates, which is what an engaged niche audience looks like. The claim is
> narrower — **subscriber count, quoted alone, carries very little information about how many
> people see a video.**

## 3. Mechanism — a stock measured against a flow

Subscribers accumulate and effectively never decay: unsubscribing is rare, and a dormant account
still counts. Reach has to be re-earned with every upload.

One of these channels stopped publishing entirely for 459 days and then returned, which separates
the two unusually cleanly.

In [ ]:
tq = d[(d.channel == "Techquickie") & (d.format == "long_video")].sort_values("date")
gap_idx = tq.date.diff().idxmax()
pos = tq.index.get_loc(gap_idx)
gap_days = tq.date.diff().max().days

before, after = tq.iloc[max(0, pos-20):pos], tq.iloc[pos:pos+20]
print(f"Techquickie published nothing for {gap_days} days "
      f"({tq.iloc[pos-1].date.date()} → {tq.iloc[pos].date.date()})\n")
print(f"  median views   before {before.views.median():>10,.0f}   "
      f"after {after.views.median():>10,.0f}   ratio {after.views.median()/before.views.median():.2f}")
print(f"  like rate      before {before.likes.sum()/before.views.sum()*100:>9.2f}%   "
      f"after {after.likes.sum()/after.views.sum()*100:>9.2f}%")
print(f"\n  subscribers lost: not observable. The channel still reports "
      f"{SUBS['Techquickie']:,},")
print(f"  more than any sibling except the flagship, while reaching the fewest people per video.")
print("\n(Interval for the view ratio, moving-block bootstrap over ±20 videos: [0.28, 0.63].")
print(" Narrower windows give unstable intervals and are not quoted — see 02 section 4b.)")

Reach fell to roughly four tenths. The subscriber side cannot be shown the same way: YouTube publishes no
subscriber history, so there is no series to plot. What can be said is weaker and still sufficient — unsubscribing
takes a deliberate act that a dormant channel never prompts, and the channel reads 4.31M today.

The blackout is an extreme version of something continuous. Any channel that slows down, changes
format, or loses algorithmic favour sheds reach immediately and sheds subscribers slowly or not at
all. The gap between the two widens, and a published subscriber figure gets less informative every
year — **in one direction only, since it can overstate but never understate**.

> The like rate held almost flat across the gap. That is *not* evidence the content was unaffected:
> as reach shrinks, the remaining audience is more self-selected, so a stable rate against a
> smaller, more committed audience is consistent with several stories. It is reported because it
> is measured, not because it settles anything.

## 4. Background — the sector is losing reach

One paragraph of context, and nothing rests on it.

Across twelve comparable technology channels collected on the same terms, per-video reach peaked
around the end of 2020 and has fallen since; ten of them are below half their peak. LMG's channels
fell further than the peer median over the last two years — point estimates put the gap at 9–23%
depending on specification — but **none of five specifications reached significance**, and with
four treated channels none could have. The direction is reported; the claim is not made.

The relevance to section 2 is only this: the gap between a subscriber stock and a reach flow widens
faster when reach is falling everywhere.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
for ch in ORDER:
    q = (d[(d.channel == ch) & (d.format == "long_video")]
           .set_index("date").views.resample("QE").median().rolling(2, min_periods=1).mean())
    q = q[(q.index >= "2018-01-01") & (q.index <= "2026-06-30")]
    ax.plot(q.index, q.values, color=COLOR[ch], lw=2, label=ch)
ax.set_yscale("log"); ax.set_ylabel("Median views per video (log)")
ax.set_title("Per-video reach, four channels", loc="left", weight="bold")
ax.legend(fontsize=8)
ax.axvspan(pd.Timestamp("2024-11-29"), pd.Timestamp("2026-03-03"), color="#000", alpha=.06)
ax.text(pd.Timestamp("2025-07-15"), ax.get_ylim()[0]*1.4, "Techquickie dark",
        ha="center", fontsize=8, color="#666")
plt.tight_layout(); plt.savefig(FIG/"reach_over_time.png", bbox_inches="tight"); plt.show()

In [ ]:
# ── for Remotion ──────────────────────────────────────────────────────────────
# Exports two things per figure: a transparent PLATE (axes, ticks, labels, titles,
# reference lines — no data marks) and a JSON of the series plus the geometry needed
# to map data coordinates onto the plate in pixels.
#
# The plate is rebuilt from scratch rather than stripped from the original, so a
# reference line can never be removed by accident. Numbers still come only from here.
import matplotlib.dates as mdates
REM = FIG/"remotion"; REM.mkdir(exist_ok=True)

def plate_geometry(fig, axes):
    """Pixel box of each axes plus its data limits — everything Remotion needs."""
    w_px, h_px = fig.get_size_inches()*fig.dpi
    out = []
    for a in axes:
        p = a.get_position()
        out.append({
            "px": {"x": p.x0*w_px, "y": (1-p.y1)*h_px, "w": p.width*w_px, "h": p.height*h_px},
            "xlim": list(a.get_xlim()), "ylim": list(a.get_ylim()),
            "xscale": a.get_xscale(), "yscale": a.get_yscale(),
        })
    return {"figure_px": {"w": w_px, "h": h_px}, "dpi": fig.dpi, "axes": out}

def save_plate(fig, name, series):
    fig.savefig(REM/f"{name}_plate.png", bbox_inches=None, transparent=True)
    geo = plate_geometry(fig, fig.axes)
    (REM/f"{name}.json").write_text(json.dumps({**geo, "series": series}, indent=1))
    print(f"  {name}_plate.png  +  {name}.json   "
          f"({geo['figure_px']['w']:.0f}x{geo['figure_px']['h']:.0f}px, {len(series)} series)")

# ---- reach_over_time --------------------------------------------------------------
from matplotlib.lines import Line2D
series, proxies = [], []
fig, ax = plt.subplots(figsize=(11, 4.2))
for ch in ORDER:
    q = (d[(d.channel == ch) & (d.format == "long_video")]
           .set_index("date").views.resample("QE").median().rolling(2, min_periods=1).mean())
    q = q[(q.index >= "2018-01-01") & (q.index <= "2026-06-30")]
    series.append({"name": ch, "color": COLOR[ch], "kind": "line",
                   "x": [t.strftime("%Y-%m-%d") for t in q.index],
                   "x_num": [float(mdates.date2num(t)) for t in q.index],
                   "y": [float(v) for v in q.values]})
    ax.plot(q.index, q.values, alpha=0)          # sets the limits, draws nothing
    proxies.append(Line2D([], [], color=COLOR[ch], lw=2, label=ch))
ax.legend(handles=proxies, fontsize=8)
ax.set_yscale("log"); ax.set_ylabel("Median views per video (log)")
ax.set_title("Per-video reach, four channels", loc="left", weight="bold")
ax.axvspan(pd.Timestamp("2024-11-29"), pd.Timestamp("2026-03-03"), color="#000", alpha=.06)
ax.text(pd.Timestamp("2025-07-15"), ax.get_ylim()[0]*1.4, "Techquickie dark",
        ha="center", fontsize=8, color="#666")
plt.tight_layout()
save_plate(fig, "reach_over_time", series); plt.close(fig)

## 5. What the attention supports

The wording here is deliberate: **exposure**, not impact. The storefront endpoint carries prices
and stock status but no units sold, so no revenue claim can be made.

In [ ]:
sku = pd.read_csv(sorted(glob.glob(str(DATA/"lttstore_skus_*.csv")))[-1])
cat = (sku.groupby("product_type")
          .agg(products=("handle", "nunique"), variants=("sku", "size"),
               median_price=("price", "median"),
               oos_pct=("available", lambda s: round((1-s.mean())*100, 1)))
          .sort_values("products", ascending=False))
cat["variant_share"] = (cat.variants/cat.variants.sum()).map("{:.0%}".format)
print(cat.to_string())

prod = sku.groupby("handle").available.any()
print(f"\n{sku.handle.nunique()} products on sale · {len(sku)} variants")
print(f"Out of stock: {(~sku.available).mean():.0%} of variants, {(~prod).mean():.0%} of products")

**The misalignment.** Tools — screwdrivers, bits, cable management — map directly onto what the
flagship makes, and they are a small category. **Apparel is the largest category by both products
and variants, and it corresponds to no content topic at all.** It is brand merchandise: it sells
on affiliation rather than on anything demonstrated in a video, which makes it the part of the
catalogue most dependent on continuing brand exposure. It also carries the highest out-of-stock
rate.

So the largest share of the storefront rests on the asset that section 2 shows is mismeasured by
the metric everyone quotes, and section 4 shows is shrinking across the sector.

This is an observation about structure, not a forecast. Without units sold there is no way to know
how much revenue any of it represents.

## 6. What I could not establish

**Content versus distribution.** Content sits upstream of distribution — what gets made drives
click-through and retention, which drive how far a video is pushed. Separating them needs
click-through rate, average view duration and retention, and all three are visible only to the
channel owner. Three attempts at a content signal, and what each showed:

- **Titles** explain 4–10% of relative performance. That is a fact about titles. A nine-word label
  is not the content, and an earlier draft wrongly read the low R² as evidence that content does
  not matter.
- **Descriptions** are available for 100% of the target set, but a median of 64% of their lines
  match promotional patterns — sponsor reads, affiliate links, social handles. The substantive part
  is one or two sentences. They are a monetisation surface, not a content summary.
- **Transcripts** are the real content and are collectable, but only after changing network; the
  first attempt was refused after 160 requests. See `04`.

**Revenue.** No units sold anywhere in the public data.

**Floatplane.** Pricing is public, scale is not. *(`06` briefly claimed to close this question on the grounds that Floatplane promotion had been removed from descriptions outright. `08` shows that was a URL-shortener migration — promotion never stopped, and it appears in 100% of 2026 uploads. The question below is fully open.)* If the company is deliberately moving value behind
its own paywall, falling YouTube reach would be partly by design rather than by failure — which
would invert the reading in section 5. Output held roughly flat across the period, which argues
against wholesale migration, but does not settle it.

**Whether the four channels should be run this way.** Section 2 shows they differ. It does not
show the difference is a mistake.

---

## Summary

Inside one company, over one window, median views per thousand subscribers ranges from 48 to 214.
The channel with 16.9 million subscribers ranks third of four; the one with 1.98 million ranks
first. Subscriber count, the figure quoted almost everywhere a channel is described, carries very
little information about how many people watch.

The mechanism is that a stock is being used to describe a flow. Subscribers accumulate and do not
decay; reach must be re-earned every upload. One channel here went dark for 459 days and returned to
roughly four tenths of its former reach, with no observable movement in the subscriber figure.

The gap keeps widening because sector reach keeps falling — per-video reach across the twelve comparable channels peaked
around the end of 2020, and ten of the twelve now sit below half their peak.

And the largest category in the company's storefront is the one least connected to any particular
video and most dependent on continuing brand exposure.